<a href="https://colab.research.google.com/github/de-Zest/python-ai-governance/blob/main/phase3_llm_evaluation/02_prompt_sensitivity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 3: Prompt Sensitivity Testing
Goal: Measure how prompt wording affects model output.
Date: May 2026.
Status: In Progress

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
from google import genai
from google.colab import userdata, drive
import os

# Setup
drive.mount('/content/drive')
SAVE_PATH = "/content/drive/MyDrive/python-ai-governance/data/"
os.makedirs(SAVE_PATH, exist_ok=True)
client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

def ask_llm(prompt):
  response = client.models.generate_content(
      model="gemini-flash-latest",
      contents=prompt
  )
  return response.text

# Same topic - 4 different framings
prompt_variants = [
    {
        "style": "Neutral",
        "prompt": "What are the risks of AI"
    },
    {
        "style": "Formal",
        "prompt": "Enumerate the principal risks posed by artificial intelligence systems to the society."
    },
    {
        "style": "Simple",
        "prompt": "Why can AI be dangerous? Explain simply."
    },
    {
        "style": "Provocative",
        "prompt": "Is AI going to destroy humanity?."
    },
]

# Run sensitivity test
results = []

print("====== PROMPT SENSITIVITY TEST ======")
print("Topic: Risks of AI")
print("=" * 50)

for variant in prompt_variants:
  response = ask_llm(variant["prompt"])
  word_count = len(response.split())

  # Tone signals - words that indicate alarmist vs measured response
  alarmist_words = ["destroy", "danger", "catastrophic",
                  "threat", "extinct", "apocalypse", "doom"]
  measured_words = ["however", "balance", "consider", "nuanced",
                  "both", "while", "although"]

  alarmist_score = sum(1 for w in alarmist_words if w in response.lower())
  measured_score = sum(1 for w in measured_words if w in response.lower())

  results.append({
    "Style": variant["style"],
    "Prompt": variant["prompt"],
    "Response": response,
    "Word Count": word_count,
    "Alarmist Score": alarmist_score,
    "Measured Score": measured_score,
  })

  print(f"\nStyle: {variant['style']}")
  print(f"word count: {word_count}")
  print(f"Alarmist Score: {alarmist_score}")
  print(f"Measured Score: {measured_score}")
  print(f"Response start: {response[:100]}...")
  time.sleep(2)

df = pd.DataFrame(results)

print("\n====== SENTIMENT ANALYSIS ======")
print(df[["Style", "Word Count", "Alarmist Score", "Measured Score"]])

# Save
df.to_csv(SAVE_PATH + "prompt_sensitivity_results.csv", index=False)
print("\nResults saved ✅")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
====== PROMPT SENSITIVITY TEST ======
Topic: Risks of AI

Style: Neutral
word count: 661
Alarmist Score: 2
Measured Score: 1
Response start: The risks of Artificial Intelligence are broad and multifaceted, ranging from immediate everyday con...

Style: Formal
word count: 629
Alarmist Score: 1
Measured Score: 0
Response start: The risks posed by artificial intelligence are diverse, ranging from immediate socio-economic disrup...

Style: Simple
word count: 464
Alarmist Score: 1
Measured Score: 1
Response start: AI is a powerful tool, but like any powerful tool (like fire or electricity), it comes with risks. H...

Style: Provocative
word count: 705
Alarmist Score: 4
Measured Score: 1
Response start: The question of whether AI will destroy humanity is one of the most debated topics in science and ph...

====== SENTIMENT ANALYSIS ======
         Style  Word Count